In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs/datasets").is_dir())

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# если понадобятся подписи — ок
from features.catalog import display_ru  # по желанию

PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "data" / "models"
OUT = ROOT / "data" / "artifacts" / "forecast"  # лучше отдельная папка, не training
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
HORIZON = 64

In [5]:
from catboost import CatBoostRegressor
import json

model = CatBoostRegressor()
model.load_model(str(MODELS / "catboost_fe.cbm"))

feat_path = MODELS / "catboost_fe_features.json"
if feat_path.exists():
    meta = json.loads(feat_path.read_text(encoding="utf-8"))
    columns = meta["features"]
    cats = meta["categorical"]
else:
    exp = json.loads((ROOT / "data/artifacts/training/catboost_fe_experiment.json").read_text(encoding="utf-8"))
    columns = exp["features"]
    cats = [c for c in columns if c in json.loads((PROCESSED / "feature_manifest.json").read_text(encoding="utf-8"))["feature_sets"]["fe"]["categorical"]]

print("trees", model.tree_count_, "n_features", len(columns))

trees 221 n_features 50


In [6]:
parts = []
for name in ["train", "valid", "test"]:
    p = PROCESSED / f"{name}.parquet"
    if p.exists():
        parts.append(pd.read_parquet(p, columns=["date", "pair_id", "quantity"]))
hist = pd.concat(parts, ignore_index=True)
hist["date"] = pd.to_datetime(hist["date"])

sample = pd.read_parquet(PROCESSED / "test.parquet", columns=None)
print(sample.columns.tolist()[:30])

['date', 'pair_id', 'quantity', 'warehouse_name', 'article', 'category_lvl_1', 'category_lvl_2', 'characteristic', 'article_warehouse', 'article_dow', 'warehouse_dow', 'dow', 'month', 'day_of_month', 'day_of_year', 'week_of_year', 'year', 'is_weekend', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'pair_age_days', 'metadata_known', 'qty_lag_1', 'qty_lag_7', 'qty_lag_14', 'qty_lag_28', 'qty_roll_mean_7', 'qty_roll_std_7']


In [7]:
cols = ["date", "warehouse_name", "article", "quantity"]
parts = [pd.read_parquet(PROCESSED / f"{n}.parquet", columns=cols) for n in ["train", "valid", "test"]]
hist = pd.concat(parts, ignore_index=True)
hist["date"] = pd.to_datetime(hist["date"])

last_date = hist["date"].max()
forecast_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=HORIZON, freq="D")

#среднее за последние 7 дней с фактом по каждой паре
last7 = hist[hist["date"] > last_date - pd.Timedelta(days=7)]
pair_mean = last7.groupby(["warehouse_name", "article"], as_index=False)["quantity"].mean()
pair_mean = pair_mean.rename(columns={"quantity": "quantity"})

#пары, которых не было в last7 — среднее за всё время
all_mean = hist.groupby(["warehouse_name", "article"], as_index=False)["quantity"].mean()
pair_mean = all_mean.merge(
    pair_mean, on=["warehouse_name", "article"], how="left", suffixes=("_all", "")
)
pair_mean["quantity"] = pair_mean["quantity"].fillna(pair_mean["quantity_all"]).clip(lower=0)
pair_mean = pair_mean[["warehouse_name", "article", "quantity"]]

#декартово: пары × 64 даты
pair_mean["key"] = 1
dates = pd.DataFrame({"date": forecast_dates, "key": 1})
sub = pair_mean.merge(dates, on="key").drop(columns="key")
sub["date"] = sub["date"].dt.strftime("%Y-%m-%d")
sub = sub[["date", "warehouse_name", "article", "quantity"]]

path = OUT / "submission_naive_mean7.csv"
sub.to_csv(path, index=False)
print(path, len(sub), sub["date"].nunique(), sub["quantity"].min())
assert sub["quantity"].isna().sum() == 0
assert (sub["quantity"] >= 0).all()

C:\Users\arllr\OneDrive\Desktop\sbertech-hackathon-main\data\artifacts\forecast\submission_naive_mean7.csv 561792 64 0.0


In [8]:
def check_submission(df: pd.DataFrame, n_pairs: int | None = None):
    need = {"date", "warehouse_name", "article", "quantity"}
    assert need <= set(df.columns), df.columns
    assert df["quantity"].isna().sum() == 0
    assert (df["quantity"] >= 0).all()
    d = pd.to_datetime(df["date"])
    assert d.nunique() == HORIZON
    assert (d.max() - d.min()).days == HORIZON - 1
    if n_pairs is not None:
        assert len(df) == n_pairs * HORIZON
    print("OK", len(df), "rows", d.min().date(), "→", d.max().date())

check_submission(sub, n_pairs=sub.groupby(["warehouse_name", "article"]).ngroups)

OK 561792 rows 2026-08-01 → 2026-10-03


In [9]:
model_path = MODELS / "catboost_fe.cbm"
feat_path = MODELS / "catboost_fe_features.json"
assert model_path.exists(), "Сначала дождись окончания 02_catboost"
meta = json.loads(feat_path.read_text(encoding="utf-8"))
columns = meta["features"]
cats = meta["categorical"]
model = CatBoostRegressor()
model.load_model(str(model_path))
print("loaded", len(columns), "features", model.tree_count_)

loaded 50 features 221


In [10]:
# TODO: скопировать из features/ функцию, которая из истории пары
# до даты t (не включая продажи t) собирает одну строку FEATURES

def predict_recursive(model, columns, cats, hist, pairs, forecast_dates):
    """hist: date, warehouse_name, article, quantity (+ нужное для фич)."""
    raise NotImplementedError("нужен build_features из feature engineering")

In [11]:
# TODO: скопировать из features/ функцию, которая из истории пары
# до даты t (не включая продажи t) собирает одну строку FEATURES

def predict_recursive(model, columns, cats, hist, pairs, forecast_dates):
    """hist: date, warehouse_name, article, quantity (+ нужное для фич)."""
    raise NotImplementedError("нужен build_features из feature engineering")

In [12]:
test = pd.read_parquet(PROCESSED / "test.parquet")
for c in cats:
    if c in test.columns:
        test[c] = test[c].astype(str).fillna("__MISSING__")

missing = [c for c in columns if c not in test.columns]
assert not missing, missing

pred = np.maximum(model.predict(test[columns]), 0)
y = test["quantity"].to_numpy(dtype=float)
mae = np.mean(np.abs(pred - y))
print("test MAE", round(mae, 4), "mean pred", round(pred.mean(), 4), "mean y", round(y.mean(), 4))

test MAE 0.2865 mean pred 0.2398 mean y 0.3997


In [13]:
cols_need = list({
    "date", "warehouse_name", "article", "quantity", "pair_id",
    *columns
})
# какие-то колонок может не быть во всех split — возьмём пересечение
def load_split(name):
    df = pd.read_parquet(PROCESSED / f"{name}.parquet")
    use = [c for c in cols_need if c in df.columns]
    return df[use].copy()

hist = pd.concat([load_split(n) for n in ["train", "valid", "test"]], ignore_index=True)
hist["date"] = pd.to_datetime(hist["date"])
hist = hist.sort_values(["warehouse_name", "article", "date"]).reset_index(drop=True)

last_date = hist["date"].max()
forecast_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=HORIZON, freq="D")

# последняя известная строка по паре (категории, stock-фичи и т.д. замораживаем)
last_rows = hist.loc[hist.groupby(["warehouse_name", "article"], sort=False)["date"].idxmax()].copy()
last_rows = last_rows.set_index(["warehouse_name", "article"])

print("last_date", last_date.date(), "pairs", len(last_rows), "horizon", HORIZON)

last_date 2026-07-31 pairs 8778 horizon 64


In [14]:
def calendar_features(date, first_seen, metadata_known):
    date = pd.Timestamp(date)
    first_seen = pd.Timestamp(first_seen)
    dow = int(date.dayofweek)
    month = int(date.month)
    return {
        "dow": dow,
        "month": month,
        "day_of_month": int(date.day),
        "day_of_year": int(date.dayofyear),
        "week_of_year": int(date.isocalendar().week),
        "year": int(date.year),
        "is_weekend": int(dow >= 5),
        "dow_sin": np.float32(np.sin(2 * np.pi * dow / 7)),
        "dow_cos": np.float32(np.cos(2 * np.pi * dow / 7)),
        "month_sin": np.float32(np.sin(2 * np.pi * (month - 1) / 12)),
        "month_cos": np.float32(np.cos(2 * np.pi * (month - 1) / 12)),
        "pair_age_days": np.int16((date - first_seen).days),
        "metadata_known": np.int8(metadata_known),
    }

def history_features(qty: pd.Series):
    """qty — series quantity по времени, индекс 0..n-1, последний = вчера относительно прогноза."""
    q = qty.astype("float64")
    out = {}
    for lag in [1, 7, 14, 28]:
        out[f"qty_lag_{lag}"] = np.float32(q.iloc[-lag]) if len(q) >= lag else np.float32(np.nan)
    # rolling по «вчерашнему» ряду: как shift(1) + rolling → у нас qty уже только прошлое
    for window in [7, 28]:
        w = q.iloc[-window:] if len(q) >= 1 else q
        out[f"qty_roll_mean_{window}"] = np.float32(w.mean()) if len(w) else np.float32(np.nan)
        out[f"qty_roll_std_{window}"] = np.float32(w.std(ddof=0)) if len(w) else np.float32(np.nan)
        out[f"qty_roll_max_{window}"] = np.float32(w.max()) if len(w) else np.float32(np.nan)
        out[f"qty_nonzero_share_{window}"] = np.float32((w > 0).mean()) if len(w) else np.float32(np.nan)
    out["history_missing"] = np.int8(1 if len(q) < 1 or pd.isna(out["qty_lag_1"]) else 0)
    m7 = out["qty_roll_mean_7"]
    m28 = out["qty_roll_mean_28"]
    out["qty_trend_7_28"] = np.float32(m7 / (m28 + 1) if not np.isnan(m7) and not np.isnan(m28) else np.nan)
    return out

def build_feature_row(date, wh, art, qty_series, frozen: pd.Series) -> dict:
    """frozen — последняя строка hist по паре (категории, stock, aggregate)."""
    row = {}
    # категории из frozen
    for c in ["warehouse_name", "article", "category_lvl_1", "category_lvl_2", "characteristic"]:
        if c in frozen.index:
            row[c] = str(frozen[c]) if pd.notna(frozen[c]) else "__MISSING__"
    row["warehouse_name"] = wh
    row["article"] = art

    first_seen = date - pd.Timedelta(days=int(frozen["pair_age_days"])) if "pair_age_days" in frozen.index else date
    # pair_age на last_date был frozen; на новую date пересчитаем грубо:
    if "pair_age_days" in frozen.index and pd.notna(frozen["pair_age_days"]):
        delta = (pd.Timestamp(date) - last_date).days
        age = int(frozen["pair_age_days"]) + delta
        first_seen = pd.Timestamp(date) - pd.Timedelta(days=age)
    mk = int(frozen["metadata_known"]) if "metadata_known" in frozen.index and pd.notna(frozen["metadata_known"]) else 1

    row.update(calendar_features(date, first_seen, mk))
    row.update(history_features(qty_series))

    # stock + aggregate: freeze с последней строки
    for c in columns:
        if c not in row and c in frozen.index:
            val = frozen[c]
            row[c] = val

    # crosses с актуальным dow
    row["article_warehouse"] = f"{art}|{wh}"
    row["article_dow"] = f"{art}|{row['dow']}"
    row["warehouse_dow"] = f"{wh}|{row['dow']}"

    # только нужные columns, в правильном порядке позже
    return row

In [15]:
# quantity-история по парам
qty_hist = {
    (wh, art): g.sort_values("date")["quantity"].reset_index(drop=True)
    for (wh, art), g in hist.groupby(["warehouse_name", "article"], sort=False)
}

pairs = list(last_rows.index)
rows_out = []

for i, date in enumerate(forecast_dates):
    batch = []
    keys = []
    for wh, art in pairs:
        key = (wh, art)
        frozen = last_rows.loc[key]
        q = qty_hist[key]
        feat = build_feature_row(date, wh, art, q, frozen)
        batch.append(feat)
        keys.append(key)

    X = pd.DataFrame(batch)
    for c in columns:
        if c not in X.columns:
            X[c] = np.nan
    X = X[columns]
    for c in cats:
        if c in X.columns:
            X[c] = X[c].astype(str).fillna("__MISSING__")

    pred = np.maximum(model.predict(X), 0.0)

    for key, p in zip(keys, pred):
        rows_out.append({
            "date": date.strftime("%Y-%m-%d"),
            "warehouse_name": key[0],
            "article": key[1],
            "quantity": float(p),
        })
        # рекурсия: pred становится новой «историей»
        qty_hist[key] = pd.concat([qty_hist[key], pd.Series([p])], ignore_index=True)

    if (i + 1) % 8 == 0 or i == 0:
        print(f"day {i+1}/{HORIZON}", date.date(), "mean_pred", round(float(pred.mean()), 4))

sub_cb = pd.DataFrame(rows_out)
path_cb = OUT / "submission_catboost_recursive_64.csv"
sub_cb.to_csv(path_cb, index=False)
check_submission(sub_cb, n_pairs=len(pairs))
print("saved", path_cb, "rows", len(sub_cb))

day 1/64 2026-08-01 mean_pred 0.2408
day 8/64 2026-08-08 mean_pred 0.2573
day 16/64 2026-08-16 mean_pred 0.3243
day 24/64 2026-08-24 mean_pred 0.3173
day 32/64 2026-09-01 mean_pred 0.2833
day 40/64 2026-09-09 mean_pred 0.2855
day 48/64 2026-09-17 mean_pred 0.2669
day 56/64 2026-09-25 mean_pred 0.2472
day 64/64 2026-10-03 mean_pred 0.2459
OK 561792 rows 2026-08-01 → 2026-10-03
saved C:\Users\arllr\OneDrive\Desktop\sbertech-hackathon-main\data\artifacts\forecast\submission_catboost_recursive_64.csv rows 561792
